In [0]:
%sql
USE CATALOG data_marts;

CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
consumidores_bronze_df = spark.table("data_marts.bronze.tb_customers")
geolocalizacao_bronze_df = spark.table("data_marts.bronze.tb_geolocalizacao")
itens_pedidos_bronze_df = spark.table("data_marts.bronze.tb_order_items")
pagamentos_pedidos_bronze_df = spark.table("data_marts.bronze.tb_order_payments")
avaliacoes_pedidos_bronze_df = spark.table("data_marts.bronze.tb_order_reviews")
pedidos_bronze_df = spark.table("data_marts.bronze.tb_orders")
produtos_bronze_df = spark.table("data_marts.bronze.tb_products")
vendedores_bronze_df = spark.table("data_marts.bronze.tb_sellers")
traducao_categoria_produtos_bronze_df = spark.table("data_marts.bronze.tb_product_category_name_translation")
cotacao_dolar_bronze_df = spark.table("data_marts.bronze.tb_cotacao_dolar")

In [0]:
catalogo = "data_marts"
silver_db_name = "silver"


In [0]:
# Função utilitária de qualidade de dados: conta valores nulos por coluna.
def analisar_null_coluna(tabela):
    tabela.select([

        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in tabela.columns
        
    ]).display()

In [0]:
# Inspeção da tabela bronze de consumidores antes da transformação.
consumidores_bronze_df.display()
analisar_null_coluna(consumidores_bronze_df)

In [0]:
# silver.dim_consumidores
# Origem: bronze.tb_customers


# usamos a deduplicação para tratar registros duplicados já que um mesmo customer_id pode
# aparecer múltiplas vezes na camada Bronze. A Window Function particiona
# por customer_id e ordena de forma decrescente pelo timestamp_ingestion,
# fazendo com que apenas o registro mais recentemente ingerido seja mantido.
window_spec = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

dim_consumidores = (
    consumidores_bronze_df
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)  # Mantém apenas o registro mais recente por consumidor
    .select(
        # Renomeação das colunas para português 
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_name").alias("nome_consumidor"),

        # Padronização: cidade e estado em maiúsculas 
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),

        # Outras colunas
        F.col("customer_unique_id").alias("id_unico_consumidor"),
        F.upper(F.col("customer_gender")).alias("genero"),
        F.to_date(F.col("customer_birth_date")).alias("data_nascimento"),
        F.col("customer_age").cast("int").alias("idade"),
    )
    # Remove registros sem ID — chave primária é obrigatória 
    .filter(F.col("id_consumidor").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_consumidores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.dim_consumidores")
print("✅ Tabela silver.dim_consumidores criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de pedidos antes da transformação.
pedidos_bronze_df.display()
analisar_null_coluna(pedidos_bronze_df)

In [0]:
# silver.fat_pedidos
# Origem: bronze.tb_orders


fat_pedidos = (
    pedidos_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),  

        # padronizar os valores para facilitar relatórios em PT-BR.
        # O otherwise mantém o valor original caso apareça algum status não mapeado.
        F.when(F.col("order_status") == "delivered", "entregue")
         .when(F.col("order_status") == "canceled", "cancelado")
         .when(F.col("order_status") == "shipped", "enviado")
         .when(F.col("order_status") == "processing", "processando")
         .when(F.col("order_status") == "invoiced","faturado")
         .when(F.col("order_status") == "unavailable","indisponível")
         .when(F.col("order_status") == "created","criado")
         .when(F.col("order_status") == "approved", "aprovado")
         .otherwise(F.col("order_status")).alias("status"),

        F.col("order_purchase_timestamp").alias("data_pedido"),
        F.col("order_delivered_customer_date").alias("data_entrega_real"),
        F.col("order_estimated_delivery_date").alias("data_entrega_estimada")
    )
    # Colunas derivadas de prazo de entrega 
    .withColumn("tempo_entrega_dias",
        # Dias entre a compra e a entrega real ao cliente
        F.datediff(F.col("data_entrega_real"),F.col("data_pedido"))
    )
    .withColumn("tempo_entrega_estimado_dias",
        # Dias entre a compra e a data estimada de entrega
        F.datediff(F.col("data_entrega_estimada"),F.col("data_pedido"))
    )
    .withColumn("diferenca_entrega_dias",
        # Diferença entre real e estimado: positivo = atrasado, negativo = adiantado
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn("entrega_no_prazo",
        # só faz sentido avaliar prazo se o pedido foi entregue.
        # Pedidos com status diferente de 'entregue' ficam como 'Não Entregue'.
        F.when(F.col("status") != "entregue", "Não Entregue")
         .when(F.col("data_entrega_real") <= F.col("data_entrega_estimada"), "Sim")
         .otherwise("Não")
    )
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

fat_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedidos")
print("✅ Tabela silver.fat_pedidos criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de itens de pedidos antes da transformação.
itens_pedidos_bronze_df.display()
analisar_null_coluna(itens_pedidos_bronze_df)

In [0]:
# silver.fat_itens_pedidos
# Origem: bronze.tb_order_items


fat_itens_pedidos = (
    itens_pedidos_bronze_df
    .select(
        # Renomeação das colunas para português 
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("price").cast("double").alias("preco_BRL"),
        F.col("freight_value").cast("double").alias("preco_frete"),
        F.col("shipping_limit_date").alias("data_limite_envio")
    )
    # Filtro para remover itens sem pedido ou sem sequencial de item
    .filter(F.col("id_pedido").isNotNull() & F.col("id_item").isNotNull())
    # Deduplicação pela chave composta pedido + item para evitar contagem dupla
    .dropDuplicates(["id_pedido", "id_item"])
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

fat_itens_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
print("✅ Tabela silver.fat_itens_pedidos criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de pagamentos antes da transformação.
pagamentos_pedidos_bronze_df.display()
analisar_null_coluna(pagamentos_pedidos_bronze_df)

In [0]:
# silver.fat_pagamentos_pedidos
# Origem: bronze.tb_order_payments


fat_pagamentos_pedidos = (
    pagamentos_pedidos_bronze_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),

        # Tradução do meio de pagamento de inglês para português.
        # padronizar os tipos para exibição em relatórios PT-BR.
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
         .when(F.col("payment_type") == "boleto", "Boleto")
         .when(F.col("payment_type") == "voucher", "Voucher")
         .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
         .when(F.col("payment_type") == "not_defined", "Não Definido")
         .otherwise(F.col("payment_type")).alias("tipo_pagamento"),

        F.col("payment_installments").cast("int").alias("numero_parcelas"),
        F.col("payment_value").cast("double").alias("valor_pagamento_BRL")
    )
    .filter(F.col("id_pedido").isNotNull())
    # Deduplicação por chave composta, já que um pedido pode ter múltiplos pagamentos sequenciais
    .dropDuplicates(["id_pedido", "sequencial_pagamento"])
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

fat_pagamentos_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")
print("✅ Tabela silver.fat_pagamentos_pedidos criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de avaliações antes da transformação.
avaliacoes_pedidos_bronze_df.display()
analisar_null_coluna(avaliacoes_pedidos_bronze_df)

In [0]:
# silver.fat_avaliacoes_pedidos
# Origem: bronze.tb_order_reviews

fat_avaliacoes_pedidos = (
    avaliacoes_pedidos_bronze_df

    # try_cast e try_to_timestamp evitam que valores
    # corrompidos ou em formato inválido quebrem o processamento do pipeline.
    .withColumn("nota_int", F.expr("try_cast(review_score as int)"))
    .withColumn("data_criacao_dt", F.try_to_timestamp(F.col("review_creation_date")))

    # Removendo registros sem nota válida, sem data de criação ou sem pedido vinculado.
    # Avaliações sem pedido  não têm valor para a analise
    .filter(
        F.col("nota_int").isNotNull() &
        F.col("data_criacao_dt").isNotNull() &
        F.col("order_id").isNotNull()
    )
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("nota_int").alias("nota_avaliacao"),

        # títulos e comentários vazios recebem texto padrão
        # para evitar NULLs que atrapalhem as contagens e analises.
        F.coalesce(F.col("review_comment_title"),   F.lit("Sem título")   ).alias("titulo_avaliacao"),
        F.coalesce(F.col("review_comment_message"), F.lit("Sem comentário")).alias("comentario_avaliacao"),

        F.col("data_criacao_dt").alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    )
    # Remove avaliações com data de criação no futuro — dados inválidos ou de teste
    .filter(F.col("data_criacao_avaliacao") <= F.current_timestamp())
    .dropDuplicates(["id_avaliacao"])
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

fat_avaliacoes_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")
print("✅ Tabela silver.fat_avaliacoes_pedidos criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de produtos antes da transformação.
produtos_bronze_df.display()
analisar_null_coluna(produtos_bronze_df)

In [0]:
# silver.dim_produtos
# Origem: bronze.tb_products


# usamos a deduplicação para  garantir que apenas o registro mais recente de cada produto seja mantido.
window_spec = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

dim_produtos = (
    produtos_bronze_df
    .withColumn("row_number", F.row_number().over(window_spec))
    .filter(F.col("row_number") == 1)
    .select(
        F.col("product_id").alias("id_produto"),
        # coalesce para produtos sem nome cadastrado, para evitar NULLs 
        F.coalesce(F.col("product_name"), F.lit("Produto sem nome")).alias("nome_produto"),
        F.coalesce(F.col("product_category_name"),F.lit("não informado")  ).alias("categoria_produto"),

        # Dimensões físicas do produto com tipagem explícita
        F.col("product_weight_g").cast("double").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("double").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("double").alias("altura_centimetros"),
        F.col("product_width_cm").cast("double").alias("largura_centimetros"),
        F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
        F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
    )
    .filter(F.col("id_produto").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_produtos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.dim_produtos")
print("✅ Tabela silver.dim_produtos criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de vendedores antes da transformação.
vendedores_bronze_df.display()
analisar_null_coluna(vendedores_bronze_df)

In [0]:
# silver.dim_vendedores
# Origem: bronze.tb_sellers

# usamos para particionar por seller_id e mantendo o registro mais recente.
window_spec_sellers = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

dim_vendedores = (
    vendedores_bronze_df
    .withColumn("row_number", F.row_number().over(window_spec_sellers))
    .filter(F.col("row_number") == 1)
    .select(

        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_name").alias("nome_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),

        # Padronização: cidade e estado em maiúsculas para uniformidade analítica
        
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
        F.col("seller_registration_date").alias("data_registro"),
    )
    .filter(F.col("id_vendedor").isNotNull())
    .withColumn("data_ingestao", F.current_timestamp())
)

dim_vendedores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.dim_vendedores")
print("✅ Tabela silver.dim_vendedores criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de tradução de categorias antes da transformação.
traducao_categoria_produtos_bronze_df.display()
analisar_null_coluna(traducao_categoria_produtos_bronze_df)

In [0]:
# silver.dim_categoria_produtos_traducao
# Origem: bronze.tb_product_category_name_translation

# Tabela de apoio que permite exibir categorias em inglês
# nos dashboards internacionais sem alterar as tabelas fato.
dim_categoria_produtos_traducao = (
    traducao_categoria_produtos_bronze_df
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en")
    )
    .withColumn("nome_produto_pt", F.trim(F.col("nome_produto_pt")))
    .withColumn("nome_produto_en", F.trim(F.col("nome_produto_en")))
    .dropDuplicates(["nome_produto_pt"])
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

dim_categoria_produtos_traducao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalogo}.{silver_db_name}.dim_categoria_produtos_traducao")
print("✅ Tabela silver.dim_categoria_produtos_traducao criada com sucesso!\n")

In [0]:
# Inspeção da tabela bronze de cotação do dólar antes da transformação.
cotacao_dolar_bronze_df.display()
analisar_null_coluna(cotacao_dolar_bronze_df)

In [0]:
# silver.dim_cotacao_dolar
# Origem: bronze.tb_cotacao_dolar (API Banco Central)

# A API do Banco Central (PTAX) não divulga cotação em finais de semana e feriados.
# Para garantir que todos os pedidos tenham uma cotação associada no JOIN,
# foi criado um calendário contínuo e preenchemos os dias sem cotação com o
# último valor conhecido

# Identifica o intervalo de datas disponível na bronze
intervalo = cotacao_dolar_bronze_df.select(
    F.min(F.to_date("dataHoraCotacao")).alias("data_inicio"),
    F.max(F.to_date("dataHoraCotacao")).alias("data_fim")
).collect()[0]

# Gera uma linha para cada dia do calendário, sem gaps
calendario_continuo = (
    spark.range(0, (intervalo["data_fim"] - intervalo["data_inicio"]).days + 1)
    .withColumn("data_referencia", F.date_add(F.lit(intervalo["data_inicio"]), F.col("id").cast("int")))
    .select("data_referencia")
)

cotacao_base = cotacao_dolar_bronze_df.select(
    F.to_date("dataHoraCotacao").alias("data_cotacao_api"),
    F.col("cotacaoCompra").cast("double").alias("valor_cotacao")
)

df_com_nulos = calendario_continuo.join(
    cotacao_base,
    calendario_continuo.data_referencia == cotacao_base.data_cotacao_api,
    "left"
)

# usa o último valor não-nulo (sexta-feira) para sábado e domingo.
# usando a Window para ordenar todas as linhas por data e aplica last() ignorando os NULLs.
window_ffill = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

dim_cotacao_dolar = (
    df_com_nulos
    .withColumn("valor_final", F.last("valor_cotacao", ignorenulls=True).over(window_ffill))
    .select(
        F.col("data_referencia").alias("data_cotacao"),
        F.col("valor_final").alias("valor_cotacao")
    )
    .withColumn("timestamp_ingestion", F.current_timestamp())
)

dim_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")
print("✅ Tabela silver.dim_cotacao_dolar criada com sucesso!\n")

In [0]:
# silver.fat_pedido_total  — Tabela Final da Camada Silver
# Origem: fat_pedidos + fat_pagamentos_pedidos + dim_cotacao_dolar


pedidos = spark.read.table(f"{catalogo}.{silver_db_name}.fat_pedidos")
pagamentos = spark.read.table(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")
cotacao = spark.read.table(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")

# Agrega todos os pagamentos de um mesmo pedido em um único valor total BRL
pagamentos_agrupados = (
    pagamentos
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento_BRL").alias("valor_total_pago_brl"))
)

fat_pedido_total = (
    pedidos.alias("p")
    .join(pagamentos_agrupados.alias("pag"), F.col("p.id_pedido") == F.col("pag.id_pedido"), "left")
    # usando join com cotação pela data do pedido para converter BRL → USD no dia da compra
    .join(cotacao.alias("c"), F.to_date(F.col("p.data_pedido")) == F.col("c.data_cotacao"), "left")
    .select(
        F.col("p.id_pedido"),
        F.col("p.id_consumidor"),     
        F.col("p.status"),             
        F.col("p.data_pedido"),
        F.round(F.col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        F.round(
            F.col("valor_total_pago_brl") / F.col("c.valor_cotacao"), 2
        ).alias("valor_total_pago_usd")
    )
    # Remove pedidos sem valor pago — não contribuem para análises financeiras
    .filter(F.col("valor_total_pago_brl").isNotNull())
)

fat_pedido_total.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedido_total")
print("✅ Tabela silver.fat_pedido_total gerada com conversão para USD e arredondamentos!")

In [0]:
# Otimização Física — Delta Lake OPTIMIZE + ZORDER

tabelas_fato = [
    f"{catalogo}.{silver_db_name}.fat_pedidos",
    f"{catalogo}.{silver_db_name}.fat_pedido_total",
    f"{catalogo}.{silver_db_name}.fat_itens_pedidos",
    f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos"
]

for tabela in tabelas_fato:
    print(f"Processando: {tabela}")
    try:
        spark.sql(f"OPTIMIZE {tabela} ZORDER BY (id_pedido, data_pedido)")
        print(f"✅ {tabela} otimizada com sucesso.")
    except Exception as e:
        print(f"⚠️ Nota: Tentando otimizar apenas por id_pedido para {tabela}...")
        spark.sql(f"OPTIMIZE {tabela} ZORDER BY (id_pedido)")
        print(f"✅ {tabela} otimizada apenas por id_pedido.")

print("\n✅ Manutenção do Lakehouse concluída.")